In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv("AQI_merged.csv")

'''#To load the dataset in Colab, comment out the line above and uncomment this section:
from google.colab import drive
drive.mount('/content/gdrive')

path = 'gdrive/MyDrive/SIADS_696_Milestone_II/Datasets/AQI_merged.csv'
df = pd.read_csv(path)'''

# Display the first five rows
df.head()

In [ ]:
# Number of rows and columns
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Column names
print("\nColumn names:")
print(df.columns.tolist())

# Data types
print("\nData types:")
print(df.dtypes)

In [ ]:
df = df.drop(columns=["Unnamed: 0"])

df.head()

In [ ]:
print(df.shape)

In [ ]:
missing_values = df.isnull().sum()

print(missing_values)

In [ ]:
plt.figure(figsize=(12, 6))

sns.heatmap(
    df.isnull(),
    cbar=False,
    yticklabels=False
)

plt.title("Missing Values in the AQI Dataset")
plt.xlabel("Variables")
plt.ylabel("Observations")

plt.show()

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
df[df.duplicated()].head()

In [ ]:
df.describe().T

In [ ]:
print("First year:", df["Year"].min())
print("Last year:", df["Year"].max())

print("\nNumber of observations by year:")
print(df["Year"].value_counts().sort_index())

In [ ]:
year_counts = df["Year"].value_counts().sort_index()

plt.figure(figsize=(14, 6))

plt.plot(year_counts.index, year_counts.values)

plt.title("Number of AQI Observations by Year")
plt.xlabel("Year")
plt.ylabel("Number of Observations")

plt.tight_layout()
plt.show()

In [ ]:
state_counts = df["State"].value_counts()

print(state_counts)

In [ ]:
plt.figure(figsize=(12, 10))

state_counts.sort_values().plot(kind="barh")

plt.title("Number of AQI Observations by State")
plt.xlabel("Number of Observations")
plt.ylabel("State")

plt.tight_layout()
plt.show()

In [ ]:
counties_by_state = (
    df.groupby("State")["County"]
      .nunique()
      .sort_values()
)

print(counties_by_state)

In [ ]:
plt.figure(figsize=(12, 10))

counties_by_state.plot(kind="barh")

plt.title("Number of Counties Represented in the AQI Dataset")
plt.xlabel("Number of Counties")
plt.ylabel("State")

plt.tight_layout()
plt.show()

In [ ]:
county_counts = (
    df.groupby(["State", "County"])
      .size()
      .reset_index(name="Observations")
)

county_counts.head()

In [ ]:
county_counts_sorted = county_counts.sort_values("Observations")

county_counts_sorted.head(25)

In [ ]:
limited_counties = county_counts_sorted.head(25).copy()

limited_counties["Location"] = (
    limited_counties["County"] + ", " +
    limited_counties["State"]
)

plt.figure(figsize=(12, 8))

sns.barplot(
    data=limited_counties,
    x="Observations",
    y="Location"
)

plt.title("Counties with the Fewest AQI Observations")
plt.xlabel("Number of Observations")
plt.ylabel("County")

plt.tight_layout()
plt.show()

In [ ]:
state_year_counts = (
    df.groupby(["State", "Year"])
      .size()
      .reset_index(name="Observations")
)

heatmap_data = state_year_counts.pivot(
    index="State",
    columns="Year",
    values="Observations"
)

heatmap_data = heatmap_data.fillna(0)

In [ ]:
plt.figure(figsize=(18, 14))

sns.heatmap(
    heatmap_data,
    cmap="Blues"
)

plt.title("AQI Data Coverage by State and Year")
plt.xlabel("Year")
plt.ylabel("State")

plt.tight_layout()
plt.show()

In [ ]:
aqi_variables = [
    "Max AQI",
    "90th Percentile AQI",
    "Median AQI"
]

df[aqi_variables].describe()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x="Median AQI",
    bins=50
)

plt.title("Distribution of Median AQI")
plt.xlabel("Median AQI")
plt.ylabel("Number of Observations")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df[
        ["Max AQI", "90th Percentile AQI", "Median AQI"]
    ]
)

plt.title("Distribution of AQI Measures")
plt.ylabel("AQI")

plt.tight_layout()
plt.show()

In [ ]:
df.nlargest(10, "Max AQI")[
    ["State", "County", "Year", "Max AQI"]
]

In [ ]:
category_columns = [
    "Good Days",
    "Moderate Days",
    "Unhealthy for Sensitive Groups Days",
    "Unhealthy Days",
    "Very Unhealthy Days",
    "Hazardous Days"
]

category_totals = df[category_columns].sum().sort_values(ascending=False)

print(category_totals)

In [ ]:
plt.figure(figsize=(12, 7))

category_totals.plot(kind="bar")

plt.title("Total Days by AQI Category")
plt.xlabel("AQI Category")
plt.ylabel("Number of Days")

plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
pollutant_columns = [
    "Days CO",
    "Days NO2",
    "Days Ozone",
    "Days PM2.5",
    "Days PM10"
]

pollutant_totals = df[pollutant_columns].sum().sort_values(ascending=False)

print(pollutant_totals)

In [ ]:
plt.figure(figsize=(10, 6))

pollutant_totals.plot(kind="bar")

plt.title("Total Days Measured by Pollutant")
plt.xlabel("Pollutant")
plt.ylabel("Number of Days")

plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
pollutant_data = df[pollutant_columns]

correlation_matrix = pollutant_data.corr()

print(correlation_matrix)

In [ ]:
plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Between Pollutant Measurement Days")

plt.tight_layout()
plt.show()

In [ ]:
yearly_aqi = (
    df.groupby("Year")["Median AQI"]
      .mean()
)

plt.figure(figsize=(14, 6))

plt.plot(
    yearly_aqi.index,
    yearly_aqi.values
)

plt.title("Average Median AQI Over Time")
plt.xlabel("Year")
plt.ylabel("Average Median AQI")

plt.tight_layout()
plt.show()

In [ ]:
yearly_moderate = (
    df.groupby("Year")["Percent Moderate Days"]
      .mean()
)

plt.figure(figsize=(14, 6))

plt.plot(
    yearly_moderate.index,
    yearly_moderate.values
)

plt.title("Average Percentage of Moderate AQI Days Over Time")
plt.xlabel("Year")
plt.ylabel("Percent of Days")

plt.tight_layout()
plt.show()

In [ ]:
state_aqi = (
    df.groupby("State")["Median AQI"]
      .mean()
      .sort_values()
)

plt.figure(figsize=(12, 10))

state_aqi.plot(kind="barh")

plt.title("Average Median AQI by State")
plt.xlabel("Average Median AQI")
plt.ylabel("State")

plt.tight_layout()
plt.show()

In [ ]:
state_year_aqi = (
    df.groupby(["State", "Year"])["Median AQI"]
      .mean()
      .reset_index()
)

aqi_heatmap = state_year_aqi.pivot(
    index="State",
    columns="Year",
    values="Median AQI"
)

In [ ]:
plt.figure(figsize=(18, 14))

sns.heatmap(
    aqi_heatmap,
    cmap="YlOrRd"
)

plt.title("Average Median AQI by State and Year")
plt.xlabel("Year")
plt.ylabel("State")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x="Days with AQI",
    bins=30
)

plt.title("Distribution of Days with Available AQI Data")
plt.xlabel("Days with AQI")
plt.ylabel("Number of Observations")

plt.tight_layout()
plt.show()

In [ ]:
coverage_summary = (
    df.groupby("State")
      .agg(
          Observations=("County", "size"),
          Counties=("County", "nunique"),
          First_Year=("Year", "min"),
          Last_Year=("Year", "max"),
          Avg_Days_AQI=("Days with AQI", "mean"),
          Avg_Median_AQI=("Median AQI", "mean")
      )
      .sort_values("Observations", ascending=False)
)

coverage_summary